In [63]:
import json
import math


# ============================================================
# FASTBOX DELIVERY SIMULATOR
# ============================================================

# ASSUMPTIONS:
# 1. Each package is assigned to the nearest agent based on
#    Euclidean distance from the agent to the warehouse.
#
# 2. If two agents are equally close, the agent with the
#    smaller ID is selected.
#
# 3. Packages are processed in the order given in the JSON.
#
# 4. Delivery route:
#       Agent -> Warehouse -> Destination
#
# 5. After delivering a package, the agent remains at the
#    destination of that package.
#
# 6. Efficiency:
#       Total distance / Packages delivered
#
# 7. Lower distance per package means better efficiency.
#
# 8. The input JSON may represent agents and warehouses either
#    as lists or dictionaries, so both formats are supported.


# ============================================================
# 1. READ JSON FILE
# ============================================================

file_path = r"C:\Users\91815\Desktop\Python assignment\base_case.json"

with open(file_path, "r") as f:
    data = json.load(f)


# ============================================================
# 2. NORMALIZE WAREHOUSES
# ============================================================

raw_warehouses = data["warehouses"]

if isinstance(raw_warehouses, list):

    warehouses = {
        warehouse["id"]: warehouse["location"]
        for warehouse in raw_warehouses
    }

else:

    warehouses = raw_warehouses


# ============================================================
# 3. NORMALIZE AGENTS
# ============================================================

raw_agents = data["agents"]

agents = {}

if isinstance(raw_agents, list):

    for agent in raw_agents:

        agents[agent["id"]] = {
            "position": agent["location"],
            "packages": [],
            "total_distance": 0
        }

else:

    for agent_id, location in raw_agents.items():

        agents[agent_id] = {
            "position": location,
            "packages": [],
            "total_distance": 0
        }


# ============================================================
# 4. GET PACKAGES
# ============================================================

packages = data["packages"]


# ============================================================
# 5. EUCLIDEAN DISTANCE FUNCTION
# ============================================================

def calculate_distance(point1, point2):

    return math.sqrt(
        (point1[0] - point2[0]) ** 2 +
        (point1[1] - point2[1]) ** 2
    )


# ============================================================
# 6. GET WAREHOUSE ID
# ============================================================

def get_warehouse_id(package):

    # Some input files use "warehouse_id"
    if "warehouse_id" in package:
        return package["warehouse_id"]

    # Other input files use "warehouse"
    elif "warehouse" in package:
        return package["warehouse"]

    else:
        raise KeyError(
            "Package does not contain warehouse information"
        )


# ============================================================
# 7. ASSIGN PACKAGES TO NEAREST AGENT
# ============================================================

for package in packages:

    warehouse_id = get_warehouse_id(package)

    warehouse_location = warehouses[warehouse_id]

    nearest_agent = min(
        agents.keys(),

        key=lambda agent_id: (
            calculate_distance(
                agents[agent_id]["position"],
                warehouse_location
            ),
            agent_id
        )
    )

    agents[nearest_agent]["packages"].append(package)


# ============================================================
# 8. SIMULATE DELIVERY
# ============================================================

for agent_id, agent in agents.items():

    for package in agent["packages"]:

        warehouse_id = get_warehouse_id(package)

        warehouse_location = warehouses[warehouse_id]

        destination = package["destination"]


        # Agent -> Warehouse

        distance_to_warehouse = calculate_distance(
            agent["position"],
            warehouse_location
        )

        agent["total_distance"] += distance_to_warehouse

        agent["position"] = warehouse_location


        # Warehouse -> Destination

        distance_to_destination = calculate_distance(
            agent["position"],
            destination
        )

        agent["total_distance"] += distance_to_destination

        agent["position"] = destination


# ============================================================
# 9. GENERATE REPORT
# ============================================================

report = {}

best_agent = None
best_efficiency = float("inf")


for agent_id, agent in agents.items():

    packages_delivered = len(agent["packages"])

    total_distance = round(
        agent["total_distance"],
        2
    )

    if packages_delivered > 0:

        efficiency = round(
            total_distance / packages_delivered,
            2
        )

    else:

        efficiency = 0


    report[agent_id] = {
        "packages_delivered": packages_delivered,
        "total_distance": total_distance,
        "efficiency": efficiency
    }


    # Find most efficient agent

    if packages_delivered > 0:

        if (
            best_agent is None
            or efficiency < best_efficiency
            or (
                efficiency == best_efficiency
                and agent_id < best_agent
            )
        ):

            best_agent = agent_id
            best_efficiency = efficiency


report["best_agent"] = best_agent


# ============================================================
# 10. VALIDATE PACKAGE COUNT
# ============================================================

total_packages_delivered = sum(
    info["packages_delivered"]
    for agent_id, info in report.items()
    if agent_id != "best_agent"
)


assert total_packages_delivered == len(packages), (
    "ERROR: Total delivered packages does not match "
    "total input packages."
)


# ============================================================
# 11. SAVE REPORT
# ============================================================

report_path = r"C:\Users\91815\Desktop\Python assignment\report.json"

with open(report_path, "w") as f:

    json.dump(
        report,
        f,
        indent=4
    )


# ============================================================
# 12. DISPLAY FINAL RESULT
# ============================================================

print("============================================")
print("FASTBOX SIMULATION COMPLETED SUCCESSFULLY")
print("============================================")

print()

print("Total packages:", len(packages))

print(
    "Total packages delivered:",
    total_packages_delivered
)

print(
    "Best agent:",
    best_agent
)

print()

print("FINAL REPORT:")

print(
    json.dumps(
        report,
        indent=4
    )
)

print()

print("Report saved at:")

print(report_path)

FASTBOX SIMULATION COMPLETED SUCCESSFULLY

Total packages: 5
Total packages delivered: 5
Best agent: A3

FINAL REPORT:
{
    "A1": {
        "packages_delivered": 2,
        "total_distance": 121.21,
        "efficiency": 60.6
    },
    "A2": {
        "packages_delivered": 2,
        "total_distance": 79.21,
        "efficiency": 39.6
    },
    "A3": {
        "packages_delivered": 1,
        "total_distance": 14.14,
        "efficiency": 14.14
    },
    "best_agent": "A3"
}

Report saved at:
C:\Users\91815\Desktop\Python assignment\report.json


In [64]:
import os

folder_path = r"C:\Users\91815\Desktop\Python assignment"

files = os.listdir(folder_path)

for file in files:
    print(file)

base_case.json
base_case_report.json
report.json
test_case_1.json
test_case_10.json
test_case_10_report.json
test_case_1_report.json
test_case_2.json
test_case_2_report.json
test_case_3.json
test_case_3_report.json
test_case_4.json
test_case_4_report.json
test_case_5.json
test_case_5_report.json
test_case_6.json
test_case_6_report.json
test_case_7.json
test_case_7_report.json
test_case_8.json
test_case_8_report.json
test_case_9.json
test_case_9_report.json
